## Figure 2c — observed and modeled precursor relationships

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: canonical relationships/figure02c.csv point metrics and
relationships/figure02c_stats.csv precomputed Pearson/OLS statistics.
The diagnostic product contains only MERRA-2 and the 207-spring WACCM
long integration, uses
daily pressure/calendar-day standardized EP before window means, the
final fixed-EOF NAM and its 1000-hPa AO slice, and the fixed low-25%
classification from all 230 WACCM springs. MERRA-2 is the top row and
WACCM is the bottom row. The plotting cell does not calculate a
minimum, window mean, standardization, correlation, or regression.

Outputs: figure2c_ubar_MERRA2NEWNAM.png and PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "02_diagnostics").is_dir() and (candidate / "03_plotting").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "runtime"
REPOSITORY_RUNTIME_ROOT = DEFAULT_DERIVED_ROOT.resolve()
PUBLIC_ROOT = Path("/mnt/soclim0/public_data/weiji").resolve()
PROTECTED_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        Path("/mnt/backup_ETH"),
        PUBLIC_ROOT / "B2000WCN001002_timefixed",
        PUBLIC_ROOT / "BWCN",
        PUBLIC_ROOT / "Hindcast",
        PUBLIC_ROOT / "Marina",
        PUBLIC_ROOT / "MERRA2M2I6NPANA",
        PUBLIC_ROOT / "MERRA2_Processed",
        PUBLIC_ROOT / "MLS",
        PUBLIC_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == PUBLIC_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if REPOSITORY_ROOT.is_dir() and not is_within(root, REPOSITORY_RUNTIME_ROOT):
        raise PermissionError(
            "PAPER1_DERIVED_ROOT must remain below this checkout's dedicated "
            f"runtime tree: root={root}, scope={REPOSITORY_RUNTIME_ROOT}"
        )
    for protected in PROTECTED_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored runtime tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run 02_diagnostics on STREAM2 or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
points_path = canonical_path("relationships/figure02c.csv")
stats_path = canonical_path("relationships/figure02c_stats.csv")
points = pd.read_csv(points_path)
statistics = pd.read_csv(stats_path)
point_columns = (
    "source", "segment", "event_id", "event_year", "model_year",
    "is_low25", "ep100_djf", "nam50_jfm", "nam50_jfma",
    "ao_jfma", "o3_minimum_du", "ep100_djf_z",
    "nam50_jfm_z", "nam50_jfma_z", "ao_jfma_z",
    "o3_minimum_du_z", "ep_standardization_population",
    "ep_standardization_master_count", "relationship_sample_scope",
)
stat_columns = (
    "source", "relation", "sample", "x_metric", "y_metric",
    "r", "p", "n", "slope", "intercept",
    "ep_standardization_population",
    "ep_standardization_master_count", "relationship_sample_scope",
)
require_columns(points, points_path, point_columns)
require_columns(statistics, stats_path, stat_columns)
points["is_low25"] = parse_boolean(points["is_low25"])

observed_points = points[
    points["source"].astype(str).str.lower().str.contains("merra")
]
modeled_points = points.drop(observed_points.index)
if set(modeled_points["segment"].astype(str)) != {"LONGRUN"}:
    raise ValueError("Figure 2c WACCM rows must be LONGRUN only")
for frame, expected_count, token in (
    (observed_points, 46, "merra2"),
    (modeled_points, 207, "longrun"),
):
    if set(frame["ep_standardization_master_count"].astype(int)) != {expected_count}:
        raise ValueError("Figure 2c relationship master count is wrong")
    scopes = set(frame["relationship_sample_scope"].astype(str).str.lower())
    if len(scopes) != 1 or token not in next(iter(scopes)):
        raise ValueError("Figure 2c relationship scope metadata is wrong")
for source in statistics["source"].astype(str).unique():
    frame = statistics[statistics["source"].astype(str) == source]
    expected_count = 46 if "merra" in source.lower() else 207
    token = "merra2" if expected_count == 46 else "longrun"
    if set(frame["ep_standardization_master_count"].astype(int)) != {expected_count}:
        raise ValueError("Figure 2c statistics master count is wrong")
    scopes = set(frame["relationship_sample_scope"].astype(str).str.lower())
    if len(scopes) != 1 or token not in next(iter(scopes)):
        raise ValueError("Figure 2c statistics scope metadata is wrong")


def source_mask(frame, observed):
    label = frame["source"].astype(str).str.lower()
    return label.str.contains("merra") if observed else ~label.str.contains("merra")


def one_stat(source_value, x_metric, y_metric, sample):
    rows = statistics[
        (statistics["source"].astype(str) == str(source_value))
        & (statistics["x_metric"].astype(str) == x_metric)
        & (statistics["y_metric"].astype(str) == y_metric)
        & (statistics["sample"].astype(str) == sample)
    ]
    if len(rows) != 1:
        raise ValueError(
            f"Expected one statistic for {source_value}, "
            f"{x_metric}, {y_metric}, {sample}; found {len(rows)}"
        )
    return rows.iloc[0]


def draw_panel(axis, frame, x_metric, y_metric, title, xlabel, ylabel):
    source_value = frame["source"].iloc[0]
    low = frame["is_low25"].astype(bool)
    axis.scatter(
        frame.loc[~low, x_metric], frame.loc[~low, y_metric],
        s=28, color="#6baed6", edgecolor="white",
        linewidth=0.35, alpha=0.82, label="other events",
    )
    axis.scatter(
        frame.loc[low, x_metric], frame.loc[low, y_metric],
        s=34, color="#e6550d", edgecolor="black",
        linewidth=0.35, alpha=0.90, label="fixed low25",
    )
    all_stats = one_stat(source_value, x_metric, y_metric, "all")
    x_grid = np.linspace(
        frame[x_metric].min(), frame[x_metric].max(), 100
    )
    axis.plot(
        x_grid,
        float(all_stats["intercept"]) + float(all_stats["slope"]) * x_grid,
        color="0.12", lw=1.35,
    )
    text_lines = []
    for sample, label in (
        ("all", "all"), ("low25", "low25"), ("other", "others")
    ):
        row = one_stat(source_value, x_metric, y_metric, sample)
        p_value = float(row["p"])
        p_label = "<0.001" if p_value < 0.001 else f"={p_value:.3f}"
        text_lines.append(
            f"{label}: r={float(row['r']):.2f}, "
            f"p{p_label}, N={int(row['n'])}"
        )
    axis.text(
        0.03, 0.97, "\n".join(text_lines),
        transform=axis.transAxes, ha="left", va="top", fontsize=7.2,
        bbox={
            "facecolor": "white", "edgecolor": "0.82", "alpha": 0.88
        },
    )
    axis.set_title(title, fontsize=11, fontweight="bold")
    axis.set_xlabel(xlabel)
    axis.set_ylabel(ylabel)
    axis.grid(color="0.90", lw=0.45)


source_specs = [
    ("MERRA-2", points[source_mask(points, True)]),
    ("WACCM long integration", points[source_mask(points, False)]),
]
figure, axes = plt.subplots(
    2, 3, figsize=(15.2, 8.2), constrained_layout=True
)
for row_index, (label, frame) in enumerate(source_specs):
    if frame.empty or frame["source"].nunique() != 1:
        raise ValueError(f"Unexpected source rows for {label}")
    draw_panel(
        axes[row_index, 0], frame,
        "ep100_djf_z", "nam50_jfm_z",
        "EP100 vs 50-hPa NAM",
        "DJF EP100", "JFM 50-hPa NAM",
    )
    draw_panel(
        axes[row_index, 1], frame,
        "ep100_djf_z", "o3_minimum_du",
        "EP100 vs O$_3$ minimum",
        "DJF EP100", "March–April O$_3$ minimum (DU)",
    )
    draw_panel(
        axes[row_index, 2], frame,
        "nam50_jfma_z", "ao_jfma_z",
        "50-hPa NAM vs AO",
        "JFMA 50-hPa NAM", "JFMA AO = NAM at 1000 hPa",
    )
    axes[row_index, 0].text(
        -0.25, 0.5, label, transform=axes[row_index, 0].transAxes,
        rotation=90, ha="center", va="center",
        fontsize=11.5, fontweight="bold",
    )
handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(
    handles, labels, loc="lower center", ncol=2,
    bbox_to_anchor=(0.5, -0.01), frameon=False,
)
figure.suptitle(
    "Observed and modeled precursor relationships",
    fontsize=15, fontweight="bold",
)
save_figure(figure, "figure2c_ubar_MERRA2NEWNAM")


## Figure 2d — continuous EP100-window sensitivity

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: canonical relationships/figure02d.nc containing precomputed
Pearson r, p, n, and validity for MERRA-2 and the 207-spring WACCM
long-integration window grids. The diagnostic stage standardized EP separately at
every pressure/calendar day before calculating each window mean and
kept the O3 response fixed. The plotting cell only selects the all-event
sample and reshapes the stored grid; it performs no window averaging or
correlation calculation. MERRA-2 is above WACCM.

Outputs: figA1.png and PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "02_diagnostics").is_dir() and (candidate / "03_plotting").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "runtime"
REPOSITORY_RUNTIME_ROOT = DEFAULT_DERIVED_ROOT.resolve()
PUBLIC_ROOT = Path("/mnt/soclim0/public_data/weiji").resolve()
PROTECTED_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        Path("/mnt/backup_ETH"),
        PUBLIC_ROOT / "B2000WCN001002_timefixed",
        PUBLIC_ROOT / "BWCN",
        PUBLIC_ROOT / "Hindcast",
        PUBLIC_ROOT / "Marina",
        PUBLIC_ROOT / "MERRA2M2I6NPANA",
        PUBLIC_ROOT / "MERRA2_Processed",
        PUBLIC_ROOT / "MLS",
        PUBLIC_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == PUBLIC_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if REPOSITORY_ROOT.is_dir() and not is_within(root, REPOSITORY_RUNTIME_ROOT):
        raise PermissionError(
            "PAPER1_DERIVED_ROOT must remain below this checkout's dedicated "
            f"runtime tree: root={root}, scope={REPOSITORY_RUNTIME_ROOT}"
        )
    for protected in PROTECTED_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored runtime tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run 02_diagnostics on STREAM2 or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
from matplotlib.patches import Rectangle

product = load_dataset(
    "relationships/figure02d.nc", ("r", "p", "n", "valid")
)
if "WACCM_LONGRUN=207" not in str(
    product.attrs.get("ep_standardization_master_counts", "")
):
    raise ValueError("Figure 2d does not record WACCM LONGRUN N=207")
if "longrun" not in str(
    product.attrs.get("waccm_relationship_scope", "")
).lower():
    raise ValueError("Figure 2d WACCM relationship scope is not LONGRUN")
required_dims = ("source", "sample", "window_length", "start_day")
if product["r"].dims != required_dims:
    raise ValueError(f"Unexpected Figure 2d dimensions {product['r'].dims}")
sample_values = list(product["sample"].values)
sample_labels = [text_value(value) for value in sample_values]
if "all" not in sample_labels:
    raise ValueError("Figure 2d lacks the all-event sample")
all_sample = sample_values[sample_labels.index("all")]
source_values = list(product["source"].values)
source_labels = [text_value(value) for value in source_values]
observed = [
    value for value, label in zip(source_values, source_labels)
    if "merra" in label.lower()
]
modeled = [
    value for value, label in zip(source_values, source_labels)
    if "merra" not in label.lower()
]
if len(observed) != 1 or len(modeled) != 1:
    raise ValueError(f"Unexpected Figure 2d sources {source_labels}")
ordered = observed + modeled
starts = np.asarray(product["start_day"].values, dtype=float)
lengths = np.asarray(product["window_length"].values, dtype=float)
figure, axes = plt.subplots(
    2, 1, figsize=(13.2, 8.2), sharex=True,
    constrained_layout=True,
)
mappable = None
for axis, source in zip(axes, ordered):
    field = product["r"].sel(source=source, sample=all_sample)
    valid = product["valid"].sel(source=source, sample=all_sample)
    values = np.where(
        np.asarray(valid.values, dtype=bool),
        np.asarray(field.values, dtype=float),
        np.nan,
    )
    mappable = axis.pcolormesh(
        starts, lengths, values,
        cmap="RdBu_r", vmin=-1, vmax=1, shading="nearest",
    )
    if 60 in starts and 90 in lengths:
        axis.add_patch(
            Rectangle(
                (60 - 2.5, 90 - 7.5), 5, 15,
                fill=False, edgecolor="#ffd400", lw=2.7,
            )
        )
    axis.set_ylabel("Window duration (days)")
    axis.set_title(
        text_value(source), loc="left", fontweight="bold"
    )
axes[-1].set_xlabel("EP100 window start")
axes[-1].set_xticks(
    [0, 31, 61, 92, 123],
    ["1 Oct", "1 Nov", "1 Dec", "1 Jan", "1 Feb"],
)
colorbar = figure.colorbar(mappable, ax=axes, pad=0.02)
colorbar.set_label("Stored Pearson r with later O$_3$ minimum")
figure.suptitle(
    "Sensitivity to the strictly preceding EP100 window",
    fontsize=14, fontweight="bold",
)
save_figure(figure, "figA1")


## Figure 2g — discrete calendar-window sensitivity

Plot action: read the canonical processed product(s), apply display-only selection/reshaping, and render this one logical figure as PNG and PDF with one shared stem.


Inputs: canonical relationships/figure02g.csv. Every row stores a
precomputed Pearson r, p, n and OLS coefficients for one source,
relationship, calendar window, and sample. The diagnostic product is
restricted to MERRA-2 and the WACCM long integration, uses the canonical NAM/AO
and calendar-day-standardized EP products, and carries the fixed
230-spring low-25% classification. The plotting cell only arranges and
annotates stored values; it contains no statistical calculation.

Outputs: figA2.png and PDF.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

def discover_repository_root() -> Path:
    """Locate the cloned ``code`` directory from a notebook kernel."""

    candidates = (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    for candidate in candidates:
        if (candidate / "02_diagnostics").is_dir() and (candidate / "03_plotting").is_dir():
            return candidate.resolve()
    raise RuntimeError(
        "Cannot locate the Paper 1 code directory. Run the notebook from the "
        "code directory or one of its notebook subdirectories."
    )


REPOSITORY_ROOT = discover_repository_root()
DEFAULT_DERIVED_ROOT = REPOSITORY_ROOT / "runtime"
REPOSITORY_RUNTIME_ROOT = DEFAULT_DERIVED_ROOT.resolve()
PUBLIC_ROOT = Path("/mnt/soclim0/public_data/weiji").resolve()
PROTECTED_DESTINATIONS = tuple(
    path.resolve()
    for path in (
        Path("/mnt/backup_ETH"),
        PUBLIC_ROOT / "B2000WCN001002_timefixed",
        PUBLIC_ROOT / "BWCN",
        PUBLIC_ROOT / "Hindcast",
        PUBLIC_ROOT / "Marina",
        PUBLIC_ROOT / "MERRA2M2I6NPANA",
        PUBLIC_ROOT / "MERRA2_Processed",
        PUBLIC_ROOT / "MLS",
        PUBLIC_ROOT / "CO2x1SmidEmin_yBWCN_timefixed",
    )
)
PRODUCT_VERSION = "Paper1_828_repro_v1"


def is_within(candidate: Path, parent: Path) -> bool:
    return candidate == parent or parent in candidate.parents


def validate_staging_root(candidate: Path) -> Path:
    root = candidate.expanduser().resolve()
    if root == Path(root.anchor) or root == PUBLIC_ROOT:
        raise PermissionError(f"Refusing unsafe staging root: {root}")
    if REPOSITORY_ROOT.is_dir() and not is_within(root, REPOSITORY_RUNTIME_ROOT):
        raise PermissionError(
            "PAPER1_DERIVED_ROOT must remain below this checkout's dedicated "
            f"runtime tree: root={root}, scope={REPOSITORY_RUNTIME_ROOT}"
        )
    for protected in PROTECTED_DESTINATIONS:
        if is_within(root, protected) or is_within(protected, root):
            raise PermissionError(
                "Refusing staging root that overlaps protected raw/legacy "
                f"tree: root={root}, protected={protected}"
            )
    return root


def canonical_root() -> Path:
    # Return the dedicated, ignored runtime tree (or an explicit safe child).
    override = os.environ.get("PAPER1_DERIVED_ROOT")
    if override:
        return validate_staging_root(Path(override))
    return validate_staging_root(DEFAULT_DERIVED_ROOT)


DERIVED_ROOT = canonical_root()
OUTPUT_DIR = Path(
    os.environ.get("PAPER1_FIGURE_ROOT", str(DERIVED_ROOT / "figures"))
).expanduser().resolve()
if not is_within(OUTPUT_DIR, DERIVED_ROOT):
    raise PermissionError(
        f"PAPER1_FIGURE_ROOT must remain below PAPER1_DERIVED_ROOT: {OUTPUT_DIR}"
    )


def canonical_path(relative: str) -> Path:
    path = DERIVED_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing canonical Paper 1 product: {path}. "
            "Run 02_diagnostics on STREAM2 or set PAPER1_DERIVED_ROOT."
        )
    return path


def load_dataset(
    relative: str,
    required_variables: tuple[str, ...],
    *,
    require_version: bool = True,
) -> xr.Dataset:
    path = canonical_path(relative)
    with xr.open_dataset(path, decode_times=False) as opened:
        dataset = opened.load()
    missing = [name for name in required_variables if name not in dataset]
    if missing:
        raise ValueError(f"{path} is missing canonical variables {missing}")
    if require_version and dataset.attrs.get("product_version") != PRODUCT_VERSION:
        raise ValueError(
            f"{path}: product_version={dataset.attrs.get('product_version')!r}; "
            f"expected {PRODUCT_VERSION!r}"
        )
    return dataset


def require_columns(
    frame: pd.DataFrame, path: Path, columns: tuple[str, ...]
) -> None:
    missing = [name for name in columns if name not in frame]
    if missing:
        raise ValueError(f"{path} is missing canonical columns {missing}")
    if "product_version" not in frame:
        raise ValueError(f"{path} is missing product_version")
    versions = set(frame["product_version"].dropna().astype(str))
    if versions != {PRODUCT_VERSION}:
        raise ValueError(
            f"{path}: product_version values {sorted(versions)!r}; "
            f"expected only {PRODUCT_VERSION!r}"
        )


def canonical_waccm_threshold() -> float:
    # Read, but never reconstruct, the fixed low-25 threshold from 230 springs.
    path = canonical_path("ozone/waccm_master_rankings.csv")
    ranking = pd.read_csv(path)
    require_columns(
        ranking, path,
        (
            "sample_size", "low25_count", "low25_threshold_du",
            "is_low25",
        ),
    )
    if len(ranking) != 230:
        raise ValueError(f"{path}: expected exactly 230 ranked springs")
    if set(ranking["sample_size"].astype(int)) != {230}:
        raise ValueError(f"{path}: sample_size must be 230 on every row")
    if set(ranking["low25_count"].astype(int)) != {57}:
        raise ValueError(f"{path}: fixed low25 count must be floor(230/4)=57")
    thresholds = pd.to_numeric(
        ranking["low25_threshold_du"], errors="raise"
    ).unique()
    if len(thresholds) != 1 or not np.isfinite(thresholds[0]):
        raise ValueError(f"{path}: expected one finite fixed threshold")
    return float(thresholds[0])


def validate_fixed_threshold(values: pd.Series, path: Path) -> float:
    stored = pd.to_numeric(values, errors="raise").unique()
    if len(stored) != 1 or not np.isfinite(stored[0]):
        raise ValueError(f"{path}: expected one finite stored low25 threshold")
    master = canonical_waccm_threshold()
    if not np.isclose(float(stored[0]), master, rtol=0.0, atol=1e-10):
        raise ValueError(
            f"{path}: threshold {stored[0]} differs from 230-spring "
            f"master threshold {master}"
        )
    return master


def parse_boolean(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values
    if pd.api.types.is_numeric_dtype(values):
        return values.astype(int).astype(bool)
    mapping = {"true": True, "false": False, "1": True, "0": False}
    parsed = values.astype(str).str.strip().str.lower().map(mapping)
    if parsed.isna().any():
        raise ValueError(
            f"Cannot parse boolean values {values[parsed.isna()].unique()}"
        )
    return parsed.astype(bool)


def text_value(value: object) -> str:
    # Decode NetCDF byte-string coordinates without producing "b'...'" labels.
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8")
    return str(value)


def date_parts(
    date_variable: xr.DataArray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = np.asarray(date_variable.values)
    if values.ndim != 1:
        raise ValueError(f"date must be one-dimensional, found {values.shape}")
    if np.issubdtype(values.dtype, np.datetime64):
        dates = pd.DatetimeIndex(values)
        return (
            dates.year.to_numpy(dtype=int),
            dates.month.to_numpy(dtype=int),
            dates.day.to_numpy(dtype=int),
        )
    compact = values.astype(np.int64)
    return compact // 10000, (compact % 10000) // 100, compact % 100


def pressure_slice(
    values: xr.DataArray, pressure_hpa: float
) -> xr.DataArray:
    if "plev" not in values.dims:
        raise ValueError(
            f"Expected canonical plev dimension, found {values.dims}"
        )
    levels = np.asarray(values["plev"].values, dtype=float)
    if np.nanmax(levels) > 1100.0:
        raise ValueError("Canonical plev must be stored in hPa")
    selected = values.sel(plev=float(pressure_hpa), method="nearest")
    actual = float(selected["plev"])
    tolerance = max(0.6, pressure_hpa * 0.02)
    if not np.isclose(actual, pressure_hpa, rtol=0.0, atol=tolerance):
        raise ValueError(
            f"Requested {pressure_hpa} hPa; nearest level is {actual}"
        )
    return selected


def save_figure(
    figure: plt.Figure, stem: str, *, dpi: int = 300
) -> None:
    if Path(stem).name != stem or not stem:
        raise ValueError(f"Figure stem must be one safe filename: {stem!r}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    png = OUTPUT_DIR / f"{stem}.png"
    pdf = OUTPUT_DIR / f"{stem}.pdf"
    temporary_png = OUTPUT_DIR / f".{stem}.{os.getpid()}.png.tmp"
    temporary_pdf = OUTPUT_DIR / f".{stem}.{os.getpid()}.pdf.tmp"
    try:
        figure.savefig(
            temporary_png, format="png", dpi=dpi,
            bbox_inches="tight", facecolor="white",
        )
        figure.savefig(
            temporary_pdf, format="pdf", bbox_inches="tight",
            facecolor="white",
        )
        if temporary_png.stat().st_size < 1024 or temporary_pdf.stat().st_size < 1024:
            raise RuntimeError(f"Rendered output is unexpectedly small: {stem}")
        os.replace(temporary_png, png)
        os.replace(temporary_pdf, pdf)
    finally:
        plt.close(figure)
        for temporary in (temporary_png, temporary_pdf):
            if temporary.exists():
                temporary.unlink()
    print(f"saved {png}")
    print(f"saved {pdf}")
from matplotlib.patches import Rectangle

path = canonical_path("relationships/figure02g.csv")
table = pd.read_csv(path)
columns = (
    "source", "relation", "window_label", "duration", "selected",
    "sample", "x_metric", "y_metric", "r", "p", "n",
    "slope", "intercept", "ep_standardization_population",
    "ep_standardization_master_count", "relationship_sample_scope",
)
require_columns(table, path, columns)
table["selected"] = parse_boolean(table["selected"])
sources = list(dict.fromkeys(table["source"].astype(str)))
observed = [value for value in sources if "merra" in value.lower()]
modeled = [value for value in sources if "merra" not in value.lower()]
if len(observed) != 1 or len(modeled) != 1:
    raise ValueError(f"Unexpected Figure 2g sources {sources}")
for source in sources:
    subset = table[table["source"].astype(str) == source]
    expected_count = 46 if "merra" in source.lower() else 207
    token = "merra2" if expected_count == 46 else "longrun"
    if set(subset["ep_standardization_master_count"].astype(int)) != {expected_count}:
        raise ValueError("Figure 2g relationship master count is wrong")
    scopes = set(subset["relationship_sample_scope"].astype(str).str.lower())
    if len(scopes) != 1 or token not in next(iter(scopes)):
        raise ValueError("Figure 2g relationship scope metadata is wrong")
relations = list(dict.fromkeys(table["relation"].astype(str)))
if len(relations) != 3:
    raise ValueError(f"Expected three Figure 2g relations, found {relations}")


def draw_stored(axis, source, relation):
    subset = table[
        (table["source"].astype(str) == source)
        & (table["relation"].astype(str) == relation)
    ].copy()
    durations = list(dict.fromkeys(subset["duration"].astype(int)))
    labels_by_duration = {
        duration: list(
            dict.fromkeys(
                subset.loc[
                    subset["duration"].astype(int) == duration,
                    "window_label",
                ].astype(str)
            )
        )
        for duration in durations
    }
    ncols = max(len(labels) for labels in labels_by_duration.values())
    matrix = np.full((len(durations), ncols), np.nan)
    annotations = []
    for row_index, duration in enumerate(durations):
        for column_index, label in enumerate(labels_by_duration[duration]):
            rows = subset[
                (subset["duration"].astype(int) == duration)
                & (subset["window_label"].astype(str) == label)
            ]
            all_row = rows[rows["sample"].astype(str) == "all"]
            other_row = rows[rows["sample"].astype(str) == "other"]
            if len(all_row) != 1 or len(other_row) != 1:
                raise ValueError(
                    f"Missing all/other stored stats for {source}, "
                    f"{relation}, {label}"
                )
            a = all_row.iloc[0]
            b = other_row.iloc[0]
            matrix[row_index, column_index] = float(a["r"])
            annotations.append(
                (
                    row_index, column_index, label, a, b,
                    bool(a["selected"]),
                )
            )
    image = axis.imshow(
        matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto"
    )
    for row_index, column_index, label, all_row, other_row, selected in annotations:
        r_all, p_all = float(all_row["r"]), float(all_row["p"])
        r_other, p_other = float(other_row["r"]), float(other_row["p"])
        star_all = "*" if p_all < 0.05 else ""
        star_other = "*" if p_other < 0.05 else ""
        color = "white" if abs(r_all) > 0.58 else "black"
        axis.text(
            column_index, row_index - 0.20, label,
            ha="center", va="center", fontsize=7.8,
            fontweight="bold", color=color,
        )
        axis.text(
            column_index, row_index + 0.22,
            f"{r_all:+.2f}{star_all}\n({r_other:+.2f}{star_other})",
            ha="center", va="center", fontsize=7.2, color=color,
        )
        if selected:
            axis.add_patch(
                Rectangle(
                    (column_index - 0.48, row_index - 0.48),
                    0.96, 0.96, fill=False,
                    edgecolor="#ffd400", lw=2.5,
                )
            )
    axis.set_title(relation, fontsize=10.5, fontweight="bold")
    axis.set_xticks([])
    axis.set_yticks(
        range(len(durations)),
        [f"{value}-month" for value in durations],
    )
    return image


ordered_sources = observed + modeled
figure, axes = plt.subplots(
    2, 3, figsize=(15.4, 7.2), constrained_layout=True
)
image = None
for row_index, source in enumerate(ordered_sources):
    for column_index, relation in enumerate(relations):
        image = draw_stored(
            axes[row_index, column_index], source, relation
        )
    axes[row_index, 0].text(
        -0.22, 0.5, source,
        transform=axes[row_index, 0].transAxes,
        rotation=90, ha="center", va="center", fontweight="bold",
    )
colorbar = figure.colorbar(image, ax=axes, pad=0.02)
colorbar.set_label("Stored Pearson r (all events; parentheses exclude low25)")
figure.suptitle(
    "Discrete calendar-window sensitivity of precursor relationships",
    fontsize=14, fontweight="bold",
)
save_figure(figure, "figA2")
